# ForgeNovaX Kaggle GPU Worker
Run all cells after selecting **GPU T4 x2**, enabling Internet, and adding the Kaggle secret `FORGENOVAX_API_KEY`. The final cell is a disabled shutdown control.

In [ ]:
# 1. Verify the real GPU environment before installing anything.
import subprocess, json, os, sys, re, secrets, time
from pathlib import Path

subprocess.run(["nvidia-smi"], check=True)
query = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader,nounits"],
    check=True, capture_output=True, text=True,
).stdout.strip().splitlines()
gpus = []
for line in query:
    index, name, memory_mib, driver = [part.strip() for part in line.split(",", 3)]
    gpus.append({"index": int(index), "name": name, "memory_mib": int(memory_mib), "driver": driver})
print(json.dumps({
    "gpu_count": len(gpus),
    "gpus": gpus,
    "total_visible_vram_mib": sum(gpu["memory_mib"] for gpu in gpus),
    "driver_version": gpus[0]["driver"] if gpus else None,
}, indent=2))
if len(gpus) != 2 or not all("T4" in gpu["name"] for gpu in gpus):
    raise RuntimeError("""WARNING: Kaggle T4 x2 is not currently active.

Open:
Notebook Settings
→ Accelerator
→ GPU T4 x2

Then restart the session.""")


In [ ]:
# 2. Obtain the project. Override FORGENOVAX_REPO_URL only for an authorized mirror/fork.
REPO_URL = os.getenv("FORGENOVAX_REPO_URL", "https://github.com/forgenovax-ui/forgenovax-kaggle-gpu-worker.git")
REPO_DIR = Path("/kaggle/working/forgenovax-kaggle-gpu-worker")
if (Path.cwd() / "scripts" / "install.sh").exists():
    REPO_DIR = Path.cwd()
elif not (REPO_DIR / "scripts" / "install.sh").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Worker source: {REPO_DIR}")


In [ ]:
# 3. Central configuration and secure API key retrieval.
defaults = {
    "MODEL": "qwen3-coder:30b",
    "CONTEXT_LENGTH": "32768",
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_KEEP_ALIVE": "20m",
    "OLLAMA_HOST": "127.0.0.1:11434",
    "PROXY_HOST": "127.0.0.1",
    "PROXY_PORT": "8000",
    "FORGENOVAX_WORK_DIR": "/kaggle/working",
}
for name, value in defaults.items():
    os.environ.setdefault(name, value)

key_from_kaggle = False
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("FORGENOVAX_API_KEY")
    key_from_kaggle = bool(api_key)
except Exception:
    api_key = ""
if not api_key:
    api_key = secrets.token_urlsafe(48)
os.environ["FORGENOVAX_API_KEY"] = api_key
if len(api_key) < 24:
    raise RuntimeError("FORGENOVAX_API_KEY must contain at least 24 characters")
print(f"Model: {os.environ['MODEL']}")
print("API key source: Kaggle secret FORGENOVAX_API_KEY" if key_from_kaggle else "API key source: cryptographically generated temporary session key")


In [ ]:
# 4. Install and verify Ollama, cloudflared, and Python dependencies.
subprocess.run(["bash", "scripts/install.sh"], check=True, env=os.environ.copy())


In [ ]:
# 5. Start Ollama with readiness polling, then pull only the active model.
subprocess.run(["bash", "scripts/start_ollama.sh"], check=True, env=os.environ.copy())
model = os.environ["MODEL"]
try:
    subprocess.run(["ollama", "pull", model], check=True, env=os.environ.copy())
except subprocess.CalledProcessError:
    log_path = Path("/kaggle/working/ollama.log")
    if log_path.exists():
        print("\nRecent Ollama log:\n", "\n".join(log_path.read_text(errors="replace").splitlines()[-80:]))
    raise
listed = subprocess.run(["ollama", "list"], check=True, capture_output=True, text=True, env=os.environ.copy())
print(listed.stdout)
if model not in listed.stdout:
    raise RuntimeError(f"Configured model was not found after pull: {model}")


In [ ]:
# 6. Local model acceptance and actual runtime evidence.
import httpx
local_model_response = httpx.post(
    "http://127.0.0.1:11434/api/chat",
    json={
        "model": model,
        "messages": [{"role": "user", "content": "Reply with exactly: FORGENOVAX_READY"}],
        "stream": False,
        "think": False,
        "keep_alive": os.environ["OLLAMA_KEEP_ALIVE"],
        "options": {"temperature": 0, "num_ctx": int(os.environ["CONTEXT_LENGTH"])},
    },
    timeout=900,
)
local_model_response.raise_for_status()
local_content = local_model_response.json()["message"]["content"].strip()
print("Local model response:", local_content)
if local_content != "FORGENOVAX_READY":
    raise RuntimeError(f"Local model acceptance failed: {local_content!r}")
print("\n--- ollama ps (actual model placement evidence) ---")
subprocess.run(["ollama", "ps"], check=True, env=os.environ.copy())
print("\n--- nvidia-smi (actual GPU/process/memory evidence) ---")
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
# 7. Start the authenticated loopback proxy and run local security/API gates.
subprocess.run(["bash", "scripts/start_proxy.sh"], check=True, env=os.environ.copy())
proxy_url = "http://127.0.0.1:8000"
missing = httpx.get(f"{proxy_url}/v1/models", timeout=10)
invalid = httpx.get(f"{proxy_url}/v1/models", headers={"Authorization": "Bearer invalid-key"}, timeout=10)
auth_headers = {"Authorization": f"Bearer {api_key}"}
valid = httpx.get(f"{proxy_url}/v1/models", headers=auth_headers, timeout=30)
print({"missing_key": missing.status_code, "invalid_key": invalid.status_code, "valid_key": valid.status_code})
if (missing.status_code, invalid.status_code, valid.status_code) != (401, 401, 200):
    raise RuntimeError("Authentication acceptance gate failed")
openai_local = httpx.post(
    f"{proxy_url}/v1/chat/completions",
    headers=auth_headers,
    json={"model": model, "messages": [{"role": "user", "content": "Reply with exactly: OPENAI_API_READY"}], "stream": False, "think": False, "temperature": 0},
    timeout=900,
)
openai_local.raise_for_status()
openai_content = openai_local.json()["choices"][0]["message"]["content"].strip()
print("OpenAI-compatible local response:", openai_content)
if openai_content != "OPENAI_API_READY":
    raise RuntimeError(f"OpenAI-compatible local acceptance failed: {openai_content!r}")


In [ ]:
# 8. Security-preflight and start a Quick Tunnel to the proxy (never to Ollama).
subprocess.run(["bash", "scripts/start_tunnel.sh"], check=True, env=os.environ.copy())
tunnel_url = Path("/kaggle/working/tunnel_url").read_text().strip()
os.environ["FORGENOVAX_TUNNEL_URL"] = tunnel_url
print("Tunnel target verified: http://127.0.0.1:8000")
print("Public URL:", tunnel_url)


In [ ]:
# 9. Full internet → Cloudflare → auth gateway → Ollama → model acceptance.
public_health = None
for _ in range(60):
    try:
        candidate = httpx.get(f"{tunnel_url}/healthz", timeout=20, follow_redirects=True)
        if candidate.status_code == 200:
            public_health = candidate
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
if public_health is None or public_health.json() != {"ok": True, "service": "forgenovax-kaggle-ai"}:
    raise RuntimeError("Public health acceptance failed")
public_missing = httpx.get(f"{tunnel_url}/v1/models", timeout=30, follow_redirects=True)
public_models = httpx.get(f"{tunnel_url}/v1/models", headers=auth_headers, timeout=60, follow_redirects=True)
print({"public_health": public_health.status_code, "public_missing_key": public_missing.status_code, "public_models": public_models.status_code})
if public_missing.status_code != 401 or public_models.status_code != 200:
    raise RuntimeError("Public authentication/models acceptance failed")
public_chat = httpx.post(
    f"{tunnel_url}/v1/chat/completions",
    headers=auth_headers,
    json={"model": model, "messages": [{"role": "user", "content": "Reply with exactly: PUBLIC_ENDPOINT_READY"}], "stream": False, "think": False, "temperature": 0},
    timeout=900, follow_redirects=True,
)
public_chat.raise_for_status()
public_content = public_chat.json()["choices"][0]["message"]["content"].strip()
print("Public model response:", public_content)
if public_content != "PUBLIC_ENDPOINT_READY":
    raise RuntimeError(f"Public inference acceptance failed: {public_content!r}")


In [ ]:
# 10. Evidence-based health report and connection summary.
subprocess.run(["bash", "scripts/status.sh"], check=True, env=os.environ.copy())
key_summary = "Stored in Kaggle secret FORGENOVAX_API_KEY" if key_from_kaggle else api_key
print("""==================================================
FORGENOVAX KAGGLE GPU WORKER
==================================================

STATUS:
READY

MODEL:
{}

OPENAI BASE URL:
{}/v1

API KEY:
{}
{}

GPU:
T4 x2

OLLAMA:
PASS

AUTHENTICATION:
PASS

PUBLIC API:
PASS

==================================================""".format(
    model, tunnel_url, key_summary, "SAVE THIS KEY NOW. It will change when the runtime resets." if not key_from_kaggle else ""
))


In [ ]:
# Optional diagnostics: recent logs with defensive secret/header redaction.
redactions = [
    (re.compile(r"(?i)(authorization\s*[:=]\s*bearer\s+)[^\s,;]+"), r"\1[REDACTED]"),
    (re.compile(r"(?i)(bearer\s+)[A-Za-z0-9._~+/-]+"), r"\1[REDACTED]"),
    (re.compile(r"(?i)(FORGENOVAX_API_KEY\s*[:=]\s*)[^\s]+"), r"\1[REDACTED]"),
]
for log_name in ("ollama.log", "proxy.log", "cloudflared.log"):
    path = Path("/kaggle/working") / log_name
    print(f"\n--- {path} ---")
    text = "\n".join(path.read_text(errors="replace").splitlines()[-80:]) if path.exists() else "not created"
    for pattern, replacement in redactions:
        text = pattern.sub(replacement, text)
    if api_key:
        text = text.replace(api_key, "[REDACTED]")
    print(text)


In [ ]:
# Manual shutdown control. Change False to True only when you are finished.
if False:
    subprocess.run(["bash", "scripts/shutdown.sh"], check=True, env=os.environ.copy())
    print("IMPORTANT:")
    print("Also stop the Kaggle notebook session from the Kaggle interface.")
    print("Otherwise GPU quota may continue being consumed.")
